<a href="https://colab.research.google.com/github/Vadapao26/WiseWaste-Inward-Procurement-Collection-Analytics/blob/main/Inward_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# *WiseWaste-Inward-Procurement-Collection-Analytics*

In [ ]:
# ============================================================
# FINAL INWARD ANALYTICS - PRODUCTION SAFE VERSION
# - Upload multiple inward cleaned CSV files
# - Analyse EACH file separately (not combined)
# - Export one Excel workbook with separate sheets per file
# - Keep fuzzy logic in backend only
# - Cleaner Material and SVL sheet design
# ============================================================

import io
import re
import numpy as np
import pandas as pd
from difflib import SequenceMatcher
from datetime import datetime
from google.colab import files
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# ============================================================
# 1) UPLOAD FILES
# ============================================================

print("📂 Upload one or more Inward cleaned CSV files...")
uploaded = files.upload()

if not uploaded:
    raise ValueError("No files uploaded.")

# ============================================================
# 2) HELPERS
# ============================================================

def find_col(df, keywords):
    """
    Find the first column whose name contains any keyword (case-insensitive).
    """
    for col in df.columns:
        col_l = str(col).strip().lower()
        if any(k in col_l for k in keywords):
            return col
    return None

def canonical_col(name):
    s = str(name).strip().lower()
    s = s.replace("_", " ").replace("-", " ").replace("/", " ")
    s = s.replace(".", " ")
    s = s.replace("(", " ").replace(")", " ")
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

COLUMN_RENAME_MAP = {
    "inward code": "Inward Code",
    "date": "Date",
    "received material from": "Received Material From",
    "received materials from": "Received Material From",
    "vendor location": "Vendor Location",
    "source": "Source",
    "project name": "Project Name",
    "vehicle number": "Vehicle Number",
    "loading cost": "Loading Cost",
    "driver name": "Driver Name",
    "vehicle vendor name": "Vehicle Vendor Name",
    "transportation cost": "Transportation Cost",
    "additional cost": "Additional Cost",
    "total tax": "Total Tax",
    "material": "Material",
    "received quantity": "Received Quantity",
    "rate": "Rate",
    "value of received material": "Value of Received Material",
    "rejected quantity": "Rejected Quantity",
    "value of accepted material": "Value of Accepted Material",
    "net procurement cost": "Net Procurement Cost",
    "payment status": "Payment Status",
    "weighment slip": "Weighment Slip",
    "vendor invoice": "Vendor Invoice",
    "vehicle gps photo": "Vehicle GPS Photo",
    "inward record photo": "Inward Record Photo",
    "material gps photo": "Material GPS Photo",
    "source file": "Source File",
}

EXPECTED_COLS = [
    "Inward Code", "Date", "Received Material From", "Vendor Location", "Source",
    "Project Name", "Vehicle Number", "Loading Cost", "Driver Name",
    "Vehicle Vendor Name", "Transportation Cost", "Additional Cost", "Total Tax",
    "Material", "Received Quantity", "Rate", "Value of Received Material",
    "Rejected Quantity", "Value of Accepted Material", "Net Procurement Cost",
    "Payment Status", "Weighment Slip", "Vendor Invoice", "Vehicle GPS Photo",
    "Inward Record Photo", "Material GPS Photo", "Source File"
]

def to_num(series):
    return pd.to_numeric(series, errors="coerce").fillna(0)

def parse_date_safe(x):
    if pd.isna(x):
        return pd.NaT
    s = str(x).strip()
    if s in ["", "nan", "NaT", "None", "<NA>"]:
        return pd.NaT

    dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
    if pd.notna(dt):
        return dt

    known_formats = [
        "%d %b %Y",
        "%d %B %Y",
        "%d-%m-%Y",
        "%d/%m/%Y",
        "%Y-%m-%d",
        "%m/%d/%Y",
    ]
    for fmt in known_formats:
        try:
            return pd.Timestamp(datetime.strptime(s, fmt))
        except:
            pass
    return pd.NaT

def canonical_material(x):
    s = str(x).strip().lower()
    s = s.replace("_", " ").replace("-", " ")
    s = re.sub(r"[^a-z0-9\s&]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def map_major_category(material):
    cm = canonical_material(material)

    if "paper" in cm or "carton" in cm:
        return "Paper"
    if "glass" in cm:
        return "Glass"
    if "plastic" in cm or "pet" in cm or "hdpe" in cm or "ldpe" in cm or "pp" in cm or "pvc" in cm:
        return "Plastics"
    if "ewaste" in cm or "e waste" in cm:
        return "E-Waste"
    if "sanitary" in cm:
        return "Sanitary Waste"
    if "food" in cm or "garden" in cm or "compost" in cm or "wet waste" in cm:
        return "Wet Waste"
    if "c&d" in cm or "debris" in cm or "tiles" in cm or "concrete" in cm:
        return "C & D Waste"
    if "drywaste" in cm or "dry waste" in cm or "textile" in cm or "rubber" in cm or "wood" in cm or "footwear" in cm:
        return "Others"
    return "Others"

def normalise_repeated_chars(s):
    # Example: niviiii -> nivi
    return re.sub(r"(.)\1{2,}", r"\1", s)

def clean_name(x):
    s = str(x).strip().lower()
    s = re.sub(r"[^a-z\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = normalise_repeated_chars(s)
    return s

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def round_numeric(df_in, decimals=2):
    df_out = df_in.copy()
    num_cols = df_out.select_dtypes(include=[np.number]).columns
    df_out[num_cols] = df_out[num_cols].round(decimals)
    return df_out

def sort_desc(df, col="Total Quantity"):
    if col in df.columns:
        return df.sort_values(by=col, ascending=False).reset_index(drop=True)
    return df.reset_index(drop=True)

def summarise(data, group_cols):
    g = data.groupby(group_cols, dropna=False).agg(
        **{
            "Total Quantity": ("Received Quantity", "sum"),
            "Total Rejection": ("Rejected Quantity", "sum"),
            "Value of Received Material": ("Final Value of Received Material", "sum"),
            "Value of Accepted Material": ("Final Value of Accepted Material", "sum"),
            "Net Procurement Cost": ("Final Net Procurement Cost", "sum"),
            "Inward Count": ("Inward Code", pd.Series.nunique),
            "Days": ("Date", lambda x: x.dropna().dt.date.nunique())
        }
    ).reset_index()

    g["Rejection %"] = np.where(
        g["Total Quantity"] > 0,
        (g["Total Rejection"] / g["Total Quantity"]) * 100,
        0
    )

    g["Average Rate"] = np.where(
        g["Total Quantity"] > 0,
        g["Value of Received Material"] / g["Total Quantity"],
        0
    )

    g["Average Quantity per Day"] = np.where(
        g["Days"] > 0,
        g["Total Quantity"] / g["Days"],
        0
    )

    g["Average Quantity per Inward"] = np.where(
        g["Inward Count"] > 0,
        g["Total Quantity"] / g["Inward Count"],
        0
    )

    total_qty = g["Total Quantity"].sum()
    g["Share %"] = np.where(total_qty > 0, (g["Total Quantity"] / total_qty) * 100, 0)

    return sort_desc(round_numeric(g))

def safe_sheet_name(prefix, suffix, max_len=31):
    base = f"{prefix}_{suffix}"
    return base[:max_len]

def write_block(writer, sheet_name, title, df_block, startrow):
    """
    Write a simple title row, then the dataframe two rows below.
    """
    pd.DataFrame([[title]]).to_excel(
        writer, sheet_name=sheet_name, index=False, header=False, startrow=startrow
    )
    df_block.to_excel(
        writer, sheet_name=sheet_name, index=False, startrow=startrow + 2
    )
    return startrow + 2 + len(df_block) + 3

# ============================================================
# 3) PROCESS EACH FILE SEPARATELY AND EXPORT TO ONE WORKBOOK
# ============================================================

output_file = "Inward_Analytics_Output_Revised.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    for filename, content in uploaded.items():
        print(f"\n🔄 Processing file: {filename}")

        # --------------------------------------------------------
        # READ THIS FILE ONLY
        # --------------------------------------------------------
        df = pd.read_csv(io.BytesIO(content))
        df["Source File"] = filename

        # --------------------------------------------------------
        # STANDARDISE COLUMN NAMES (if the file is already clean, this is harmless)
        # --------------------------------------------------------
        df.columns = [COLUMN_RENAME_MAP.get(canonical_col(c), str(c).strip()) for c in df.columns]

        # Merge duplicate columns if created after renaming
        if not df.columns.is_unique:
            merged = pd.DataFrame(index=df.index)
            for col in df.columns.unique():
                same = df.loc[:, df.columns == col]
                if same.shape[1] == 1:
                    merged[col] = same.iloc[:, 0]
                else:
                    combined = same.iloc[:, 0].copy()
                    for i in range(1, same.shape[1]):
                        combined = combined.combine_first(same.iloc[:, i])
                    merged[col] = combined
            df = merged

        # Ensure expected output columns exist
        for col in EXPECTED_COLS:
            if col not in df.columns:
                df[col] = pd.NA

        # --------------------------------------------------------
        # DYNAMIC COLUMN DETECTION
        # --------------------------------------------------------
        material_col = "Material" if "Material" in df.columns else find_col(df, ["material"])
        source_col = "Source" if "Source" in df.columns else find_col(df, ["source"])
        vendor_col = "Received Material From" if "Received Material From" in df.columns else find_col(df, ["received material from", "vendor"])
        location_col = "Vendor Location" if "Vendor Location" in df.columns else find_col(df, ["vendor location", "location"])
        driver_col = "Driver Name" if "Driver Name" in df.columns else find_col(df, ["driver"])
        vehicle_col = "Vehicle Number" if "Vehicle Number" in df.columns else find_col(df, ["vehicle"])

        rate_col = "Rate" if "Rate" in df.columns else find_col(df, ["rate"])
        qty_col = "Received Quantity" if "Received Quantity" in df.columns else find_col(df, ["received quantity", "quantity", "qty", "weight"])
        rej_col = "Rejected Quantity" if "Rejected Quantity" in df.columns else find_col(df, ["rejected"])

        # Standard text fields
        df["Material"] = df[material_col].astype(str).str.strip() if material_col else "Unknown"
        df["Source"] = df[source_col].astype(str).str.strip() if source_col else "Unknown"
        df["Received Material From"] = df[vendor_col].astype(str).str.strip() if vendor_col else "Unknown"
        df["Vendor Location"] = df[location_col].astype(str).str.strip() if location_col else "Unknown"
        df["Driver Name"] = df[driver_col].astype(str).str.strip() if driver_col else "Unknown"
        df["Vehicle Number"] = df[vehicle_col].astype(str).str.strip() if vehicle_col else "Unknown"

        # Standard numeric fields
        df["Rate"] = to_num(df[rate_col]) if rate_col else 0
        df["Received Quantity"] = to_num(df[qty_col]) if qty_col else 0
        df["Rejected Quantity"] = to_num(df[rej_col]) if rej_col else 0

        # Optional numeric fields if present
        for optional_num_col in ["Loading Cost", "Transportation Cost", "Additional Cost", "Total Tax",
                                 "Value of Received Material", "Value of Accepted Material", "Net Procurement Cost"]:
            if optional_num_col not in df.columns:
                df[optional_num_col] = 0
            df[optional_num_col] = to_num(df[optional_num_col])

        # --------------------------------------------------------
        # DATE PARSING
        # --------------------------------------------------------
        if "Date" not in df.columns:
            df["Date"] = pd.NA

        df["Original Date Text"] = df["Date"].astype(str).str.strip()
        df["Date Parsed"] = df["Original Date Text"].apply(parse_date_safe)
        df["Date Parse Status"] = np.where(df["Date Parsed"].isna(), "Unparsed", "Parsed")

        df["Date"] = df["Date Parsed"]

        df["Date Text"] = np.where(
            df["Date"].notna(),
            df["Date"].dt.strftime("%d %b %Y"),
            df["Original Date Text"]
        )
        df["Year"] = df["Date"].dt.year
        df["Month"] = df["Date"].dt.month
        df["Month Name"] = df["Date"].dt.strftime("%b")
        df["Month-Year"] = df["Date"].dt.strftime("%b-%Y")
        df["Day"] = df["Date"].dt.day

        # --------------------------------------------------------
        # DERIVED COLUMNS
        # --------------------------------------------------------
        df["Accepted Quantity"] = (df["Received Quantity"] - df["Rejected Quantity"]).clip(lower=0)

        computed_received_value = df["Received Quantity"] * df["Rate"]
        computed_accepted_value = df["Accepted Quantity"] * df["Rate"]
        computed_net_procurement = computed_accepted_value - (
            df["Transportation Cost"] + df["Additional Cost"] + df["Loading Cost"]
        )

        # If existing value columns are present, use them; otherwise use computed
        df["Final Value of Received Material"] = np.where(
            df["Value of Received Material"].notna(),
            df["Value of Received Material"],
            computed_received_value
        )

        df["Final Value of Accepted Material"] = np.where(
            df["Value of Accepted Material"].notna(),
            df["Value of Accepted Material"],
            computed_accepted_value
        )

        df["Final Net Procurement Cost"] = np.where(
            df["Net Procurement Cost"].notna(),
            df["Net Procurement Cost"],
            computed_net_procurement
        )

        df["Rejection %"] = np.where(
            df["Received Quantity"] > 0,
            (df["Rejected Quantity"] / df["Received Quantity"]) * 100,
            0
        )

        df["Material Value Type"] = np.where(df["Rate"] > 0, "Valuable", "Non-Valuable")
        df["Major Category"] = df["Material"].apply(map_major_category)

        # --------------------------------------------------------
        # DRIVER FUZZY LOGIC (BACKEND ONLY)
        # --------------------------------------------------------
        df["Driver Name Original"] = df["Driver Name"].fillna("").astype(str).str.strip()
        df["Driver Name Clean"] = df["Driver Name Original"].apply(clean_name)

        unique_names = [n for n in df["Driver Name Clean"].unique().tolist() if n != ""]
        threshold = 0.82
        groups = []
        standard_map = {}

        for name in sorted(unique_names, key=len, reverse=True):
            matched = None
            for grp in groups:
                rep = grp[0]
                if similarity(name, rep) >= threshold:
                    matched = grp
                    break
            if matched is None:
                groups.append([name])
            else:
                matched.append(name)

        for grp in groups:
            rep = sorted(grp, key=lambda x: (-len(x), x))[0]
            for item in grp:
                standard_map[item] = rep

        df["Standardised Driver Name"] = df["Driver Name Clean"].map(standard_map).fillna(df["Driver Name Clean"])
        df["Standardised Driver Name"] = df["Standardised Driver Name"].replace("", "Unknown Driver")

        # Backend-only support tables (not exported)
        _driver_vehicle_mapping_backend = (
            df.groupby("Standardised Driver Name")["Vehicle Number"]
            .apply(lambda x: ", ".join(sorted(set([i for i in x.astype(str) if i.strip() != ""]))))
            .reset_index(name="Vehicles Used")
        )

        _driver_fuzzy_review_backend = (
            df[["Driver Name Original", "Driver Name Clean", "Standardised Driver Name"]]
            .drop_duplicates()
            .sort_values(["Standardised Driver Name", "Driver Name Original"])
            .reset_index(drop=True)
        )

        # --------------------------------------------------------
        # DOCUMENT FLAGS + EXCEPTIONS
        # --------------------------------------------------------
        doc_cols = ["Weighment Slip", "Vendor Invoice", "Vehicle GPS Photo", "Inward Record Photo", "Material GPS Photo"]
        for col in doc_cols:
            if col not in df.columns:
                df[col] = ""
            df[f"{col} Available"] = np.where(
                df[col].astype(str).str.strip().isin(["", "nan", "None", "<NA>"]),
                "No",
                "Yes"
            )

        def build_exception_flags(row, high_rejection_threshold=20):
            flags = []
            if row["Date Parse Status"] == "Unparsed":
                flags.append("Date Unparsed")
            if row["Final Net Procurement Cost"] < 0:
                flags.append("Negative Net Procurement Cost")
            if row["Rejection %"] > high_rejection_threshold:
                flags.append(f"High Rejection > {high_rejection_threshold}%")
            return " | ".join(flags) if flags else ""

        df["Exception Flags"] = df.apply(build_exception_flags, axis=1)

        # ========================================================
        # KPI SUMMARY
        # ========================================================
        total_qty = df["Received Quantity"].sum()
        total_rej = df["Rejected Quantity"].sum()
        total_days = df["Date"].dropna().dt.date.nunique()
        total_inwards = df["Inward Code"].nunique() if "Inward Code" in df.columns else len(df)

        kpi_summary = pd.DataFrame({
            "Metric": [
                "Total Quantity",
                "Total Rejection",
                "Average Rejection %",
                "Average Rate",
                "Average Quantity per Day",
                "Average Quantity per Inward",
                "Total Value of Received Material",
                "Total Value of Accepted Material",
                "Total Net Procurement Cost",
                "Total Valuable Quantity",
                "Total Non-Valuable Quantity",
                "Total Distinct Sources",
                "Total Distinct Vendors",
                "Total Distinct Locations",
                "Total Distinct Standardised Drivers",
                "Total Distinct Vehicles",
                "Rows with Negative Net Procurement Cost",
                "Rows with Unparsed Date"
            ],
            "Value": [
                total_qty,
                total_rej,
                (total_rej / total_qty * 100) if total_qty > 0 else 0,
                (df["Final Value of Received Material"].sum() / total_qty) if total_qty > 0 else 0,
                (total_qty / total_days) if total_days > 0 else 0,
                (total_qty / total_inwards) if total_inwards > 0 else 0,
                df["Final Value of Received Material"].sum(),
                df["Final Value of Accepted Material"].sum(),
                df["Final Net Procurement Cost"].sum(),
                df.loc[df["Material Value Type"] == "Valuable", "Received Quantity"].sum(),
                df.loc[df["Material Value Type"] == "Non-Valuable", "Received Quantity"].sum(),
                df["Source"].nunique(),
                df["Received Material From"].nunique(),
                df["Vendor Location"].nunique(),
                df["Standardised Driver Name"].nunique(),
                df["Vehicle Number"].nunique(),
                int((df["Final Net Procurement Cost"] < 0).sum()),
                int((df["Date Parse Status"] == "Unparsed").sum())
            ]
        })
        kpi_summary = round_numeric(kpi_summary)

        # ========================================================
        # RAW DATA ENRICHED
        # ========================================================
        raw_output_cols = [
            "Source File",
            "Inward Code",
            "Original Date Text",
            "Date Text",
            "Date Parse Status",
            "Year",
            "Month",
            "Month Name",
            "Month-Year",
            "Source",
            "Received Material From",
            "Vendor Location",
            "Project Name",
            "Vehicle Number",
            "Driver Name Original",
            "Standardised Driver Name",
            "Vehicle Vendor Name",
            "Material",
            "Major Category",
            "Material Value Type",
            "Received Quantity",
            "Rejected Quantity",
            "Accepted Quantity",
            "Rejection %",
            "Rate",
            "Final Value of Received Material",
            "Final Value of Accepted Material",
            "Loading Cost",
            "Transportation Cost",
            "Additional Cost",
            "Final Net Procurement Cost",
            "Payment Status",
            "Weighment Slip Available",
            "Vendor Invoice Available",
            "Vehicle GPS Photo Available",
            "Inward Record Photo Available",
            "Material GPS Photo Available",
            "Exception Flags"
        ]
        for col in raw_output_cols:
            if col not in df.columns:
                df[col] = pd.NA
        raw_data_enriched = round_numeric(df[raw_output_cols].copy())

        # ========================================================
        # MATERIAL ANALYSIS (3 BLOCKS)
        # ========================================================
        category_base = summarise(df, ["Major Category"])
        category_val_split = (
            df.groupby(["Major Category", "Material Value Type"], dropna=False)["Received Quantity"]
            .sum()
            .unstack(fill_value=0)
            .reset_index()
        )
        for c in ["Valuable", "Non-Valuable"]:
            if c not in category_val_split.columns:
                category_val_split[c] = 0
        category_summary = category_base.merge(category_val_split, on="Major Category", how="left")
        category_summary = category_summary.rename(columns={"Valuable": "Valuable Quantity", "Non-Valuable": "Non-Valuable Quantity"})
        category_summary["Valuable %"] = np.where(
            category_summary["Total Quantity"] > 0,
            (category_summary["Valuable Quantity"] / category_summary["Total Quantity"]) * 100,
            0
        )
        category_summary = sort_desc(round_numeric(category_summary))

        material_base = (
            df.groupby(["Major Category", "Material"], dropna=False)
            .agg(
                **{
                    "Total Quantity": ("Received Quantity", "sum"),
                    "Total Rejection": ("Rejected Quantity", "sum"),
                    "Value of Received Material": ("Final Value of Received Material", "sum"),
                    "Value of Accepted Material": ("Final Value of Accepted Material", "sum"),
                    "Net Procurement Cost": ("Final Net Procurement Cost", "sum"),
                    "Inward Count": ("Inward Code", pd.Series.nunique),
                    "Days": ("Date", lambda x: x.dropna().dt.date.nunique())
                }
            ).reset_index()
        )
        material_val_split = (
            df.groupby(["Major Category", "Material", "Material Value Type"], dropna=False)["Received Quantity"]
            .sum()
            .unstack(fill_value=0)
            .reset_index()
        )
        for c in ["Valuable", "Non-Valuable"]:
            if c not in material_val_split.columns:
                material_val_split[c] = 0
        material_summary = material_base.merge(material_val_split, on=["Major Category", "Material"], how="left")
        material_summary = material_summary.rename(columns={"Valuable": "Valuable Quantity", "Non-Valuable": "Non-Valuable Quantity"})
        material_summary["Rejection %"] = np.where(
            material_summary["Total Quantity"] > 0,
            (material_summary["Total Rejection"] / material_summary["Total Quantity"]) * 100,
            0
        )
        material_summary["Average Rate"] = np.where(
            material_summary["Total Quantity"] > 0,
            material_summary["Value of Received Material"] / material_summary["Total Quantity"],
            0
        )
        material_summary["Average Quantity per Day"] = np.where(
            material_summary["Days"] > 0,
            material_summary["Total Quantity"] / material_summary["Days"],
            0
        )
        material_summary["Average Quantity per Inward"] = np.where(
            material_summary["Inward Count"] > 0,
            material_summary["Total Quantity"] / material_summary["Inward Count"],
            0
        )
        material_summary["Valuable %"] = np.where(
            material_summary["Total Quantity"] > 0,
            (material_summary["Valuable Quantity"] / material_summary["Total Quantity"]) * 100,
            0
        )
        total_material_qty = material_summary["Total Quantity"].sum()
        material_summary["Share %"] = np.where(
            total_material_qty > 0,
            (material_summary["Total Quantity"] / total_material_qty) * 100,
            0
        )
        material_summary = sort_desc(round_numeric(material_summary))

        value_type_summary = summarise(df, ["Material Value Type"])
        value_type_summary = sort_desc(round_numeric(value_type_summary))

        # ========================================================
        # SVL ANALYSIS (3 BLOCKS)
        # ========================================================
        source_base = summarise(df, ["Source"])
        source_val_split = (
            df.groupby(["Source", "Material Value Type"], dropna=False)["Received Quantity"]
            .sum()
            .unstack(fill_value=0)
            .reset_index()
        )
        for c in ["Valuable", "Non-Valuable"]:
            if c not in source_val_split.columns:
                source_val_split[c] = 0
        source_summary = source_base.merge(source_val_split, on="Source", how="left")
        source_summary = source_summary.rename(columns={"Valuable": "Valuable Quantity", "Non-Valuable": "Non-Valuable Quantity"})
        source_summary["Valuable %"] = np.where(
            source_summary["Total Quantity"] > 0,
            (source_summary["Valuable Quantity"] / source_summary["Total Quantity"]) * 100,
            0
        )
        source_summary = sort_desc(round_numeric(source_summary))

        svl_base = summarise(df, ["Source", "Received Material From", "Vendor Location"])
        svl_val_split = (
            df.groupby(["Source", "Received Material From", "Vendor Location", "Material Value Type"], dropna=False)["Received Quantity"]
            .sum()
            .unstack(fill_value=0)
            .reset_index()
        )
        for c in ["Valuable", "Non-Valuable"]:
            if c not in svl_val_split.columns:
                svl_val_split[c] = 0
        svl_drilldown = svl_base.merge(
            svl_val_split,
            on=["Source", "Received Material From", "Vendor Location"],
            how="left"
        )
        svl_drilldown = svl_drilldown.rename(columns={"Valuable": "Valuable Quantity", "Non-Valuable": "Non-Valuable Quantity"})
        svl_drilldown["Valuable %"] = np.where(
            svl_drilldown["Total Quantity"] > 0,
            (svl_drilldown["Valuable Quantity"] / svl_drilldown["Total Quantity"]) * 100,
            0
        )
        svl_drilldown = sort_desc(round_numeric(svl_drilldown))

        # Clean hierarchy material/category mix
        source_mix = (
            df.groupby(["Source", "Major Category", "Material"], dropna=False)
            .agg(
                **{
                    "Total Quantity": ("Received Quantity", "sum"),
                    "Total Rejection": ("Rejected Quantity", "sum"),
                    "Value of Received Material": ("Final Value of Received Material", "sum"),
                    "Value of Accepted Material": ("Final Value of Accepted Material", "sum"),
                    "Net Procurement Cost": ("Final Net Procurement Cost", "sum")
                }
            ).reset_index()
        )
        source_mix["Level"] = "Source Mix"
        source_mix["Received Material From"] = ""
        source_mix["Vendor Location"] = ""

        vendor_mix = (
            df.groupby(["Received Material From", "Major Category", "Material"], dropna=False)
            .agg(
                **{
                    "Total Quantity": ("Received Quantity", "sum"),
                    "Total Rejection": ("Rejected Quantity", "sum"),
                    "Value of Received Material": ("Final Value of Received Material", "sum"),
                    "Value of Accepted Material": ("Final Value of Accepted Material", "sum"),
                    "Net Procurement Cost": ("Final Net Procurement Cost", "sum")
                }
            ).reset_index()
        )
        vendor_mix["Level"] = "Vendor Mix"
        vendor_mix["Source"] = ""
        vendor_mix["Vendor Location"] = ""

        location_mix = (
            df.groupby(["Vendor Location", "Major Category", "Material"], dropna=False)
            .agg(
                **{
                    "Total Quantity": ("Received Quantity", "sum"),
                    "Total Rejection": ("Rejected Quantity", "sum"),
                    "Value of Received Material": ("Final Value of Received Material", "sum"),
                    "Value of Accepted Material": ("Final Value of Accepted Material", "sum"),
                    "Net Procurement Cost": ("Final Net Procurement Cost", "sum")
                }
            ).reset_index()
        )
        location_mix["Level"] = "Location Mix"
        location_mix["Source"] = ""
        location_mix["Received Material From"] = ""

        svl_mix = pd.concat([source_mix, vendor_mix, location_mix], ignore_index=True, sort=False)

        mix_val_split_source = (
            df.groupby(["Source", "Major Category", "Material", "Material Value Type"], dropna=False)["Received Quantity"]
            .sum().unstack(fill_value=0).reset_index()
        )
        for c in ["Valuable", "Non-Valuable"]:
            if c not in mix_val_split_source.columns:
                mix_val_split_source[c] = 0
        mix_val_split_source["Level"] = "Source Mix"
        mix_val_split_source["Received Material From"] = ""
        mix_val_split_source["Vendor Location"] = ""

        mix_val_split_vendor = (
            df.groupby(["Received Material From", "Major Category", "Material", "Material Value Type"], dropna=False)["Received Quantity"]
            .sum().unstack(fill_value=0).reset_index()
        )
        for c in ["Valuable", "Non-Valuable"]:
            if c not in mix_val_split_vendor.columns:
                mix_val_split_vendor[c] = 0
        mix_val_split_vendor["Level"] = "Vendor Mix"
        mix_val_split_vendor["Source"] = ""
        mix_val_split_vendor["Vendor Location"] = ""

        mix_val_split_location = (
            df.groupby(["Vendor Location", "Major Category", "Material", "Material Value Type"], dropna=False)["Received Quantity"]
            .sum().unstack(fill_value=0).reset_index()
        )
        for c in ["Valuable", "Non-Valuable"]:
            if c not in mix_val_split_location.columns:
                mix_val_split_location[c] = 0
        mix_val_split_location["Level"] = "Location Mix"
        mix_val_split_location["Source"] = ""
        mix_val_split_location["Received Material From"] = ""

        mix_val_split = pd.concat(
            [mix_val_split_source, mix_val_split_vendor, mix_val_split_location],
            ignore_index=True, sort=False
        ).rename(columns={"Valuable": "Valuable Quantity", "Non-Valuable": "Non-Valuable Quantity"})

        merge_cols = ["Level", "Source", "Received Material From", "Vendor Location", "Major Category", "Material"]
        svl_mix = svl_mix.merge(mix_val_split[merge_cols + ["Valuable Quantity", "Non-Valuable Quantity"]], on=merge_cols, how="left")
        svl_mix["Valuable Quantity"] = svl_mix["Valuable Quantity"].fillna(0)
        svl_mix["Non-Valuable Quantity"] = svl_mix["Non-Valuable Quantity"].fillna(0)
        svl_mix["Valuable %"] = np.where(
            svl_mix["Total Quantity"] > 0,
            (svl_mix["Valuable Quantity"] / svl_mix["Total Quantity"]) * 100,
            0
        )
        svl_mix["Rejection %"] = np.where(
            svl_mix["Total Quantity"] > 0,
            (svl_mix["Total Rejection"] / svl_mix["Total Quantity"]) * 100,
            0
        )
        svl_mix["Average Rate"] = np.where(
            svl_mix["Total Quantity"] > 0,
            svl_mix["Value of Received Material"] / svl_mix["Total Quantity"],
            0
        )
        total_mix_qty = svl_mix["Total Quantity"].sum()
        svl_mix["Share %"] = np.where(
            total_mix_qty > 0,
            (svl_mix["Total Quantity"] / total_mix_qty) * 100,
            0
        )
        svl_mix = sort_desc(round_numeric(svl_mix))

        # ========================================================
        # DRIVER ANALYSIS (sorted)
        # ========================================================
        driver_total = summarise(df, ["Standardised Driver Name"]).rename(columns={"Standardised Driver Name": "Driver"})

        driver_valuable = (
            df[df["Material Value Type"] == "Valuable"]
            .groupby("Standardised Driver Name", dropna=False)["Received Quantity"]
            .sum()
            .reset_index(name="Valuable Quantity")
            .rename(columns={"Standardised Driver Name": "Driver"})
        )

        driver_nonvaluable = (
            df[df["Material Value Type"] == "Non-Valuable"]
            .groupby("Standardised Driver Name", dropna=False)["Received Quantity"]
            .sum()
            .reset_index(name="Non-Valuable Quantity")
            .rename(columns={"Standardised Driver Name": "Driver"})
        )

        driver_original_names = (
            df.groupby("Standardised Driver Name")["Driver Name Original"]
            .apply(lambda x: ", ".join(sorted(set([i for i in x.astype(str) if i.strip() != ""]))))
            .reset_index(name="Original Driver Names")
            .rename(columns={"Standardised Driver Name": "Driver"})
        )

        driver_vehicles = (
            df.groupby("Standardised Driver Name")["Vehicle Number"]
            .apply(lambda x: ", ".join(sorted(set([i for i in x.astype(str) if i.strip() != ""]))))
            .reset_index(name="Vehicles Used")
            .rename(columns={"Standardised Driver Name": "Driver"})
        )

        driver_analysis = (
            driver_total
            .merge(driver_valuable, on="Driver", how="left")
            .merge(driver_nonvaluable, on="Driver", how="left")
            .merge(driver_original_names, on="Driver", how="left")
            .merge(driver_vehicles, on="Driver", how="left")
        )

        driver_analysis["Valuable Quantity"] = driver_analysis["Valuable Quantity"].fillna(0)
        driver_analysis["Non-Valuable Quantity"] = driver_analysis["Non-Valuable Quantity"].fillna(0)
        driver_analysis["Valuable %"] = np.where(
            driver_analysis["Total Quantity"] > 0,
            (driver_analysis["Valuable Quantity"] / driver_analysis["Total Quantity"]) * 100,
            0
        )
        driver_analysis = sort_desc(round_numeric(driver_analysis))

        # ========================================================
        # VEHICLE ANALYSIS (sorted)
        # ========================================================
        vehicle_total = summarise(df, ["Vehicle Number"])

        vehicle_valuable = (
            df[df["Material Value Type"] == "Valuable"]
            .groupby("Vehicle Number", dropna=False)["Received Quantity"]
            .sum()
            .reset_index(name="Valuable Quantity")
        )

        vehicle_nonvaluable = (
            df[df["Material Value Type"] == "Non-Valuable"]
            .groupby("Vehicle Number", dropna=False)["Received Quantity"]
            .sum()
            .reset_index(name="Non-Valuable Quantity")
        )

        vehicle_drivers = (
            df.groupby("Vehicle Number")["Standardised Driver Name"]
            .apply(lambda x: ", ".join(sorted(set([i for i in x.astype(str) if i.strip() != ""]))))
            .reset_index(name="Drivers Used")
        )

        vehicle_analysis = (
            vehicle_total
            .merge(vehicle_valuable, on="Vehicle Number", how="left")
            .merge(vehicle_nonvaluable, on="Vehicle Number", how="left")
            .merge(vehicle_drivers, on="Vehicle Number", how="left")
        )

        vehicle_analysis["Valuable Quantity"] = vehicle_analysis["Valuable Quantity"].fillna(0)
        vehicle_analysis["Non-Valuable Quantity"] = vehicle_analysis["Non-Valuable Quantity"].fillna(0)
        vehicle_analysis["Valuable %"] = np.where(
            vehicle_analysis["Total Quantity"] > 0,
            (vehicle_analysis["Valuable Quantity"] / vehicle_analysis["Total Quantity"]) * 100,
            0
        )
        vehicle_analysis = sort_desc(round_numeric(vehicle_analysis))

        # ========================================================
        # WRITE SHEETS FOR THIS FILE
        # ========================================================
        prefix = re.sub(r"[^A-Za-z0-9]", "_", filename.split(".")[0])[:18]

        # KPI
        kpi_summary.to_excel(writer, sheet_name=safe_sheet_name(prefix, "KPI"), index=False)

        # Raw Data
        raw_data_enriched.to_excel(writer, sheet_name=safe_sheet_name(prefix, "Raw"), index=False)

        # Material (3 blocks)
        material_sheet = safe_sheet_name(prefix, "Material")
        row = 0
        row = write_block(writer, material_sheet, "Major Category Summary", category_summary, row)
        row = write_block(writer, material_sheet, "Material Summary", material_summary, row)
        row = write_block(writer, material_sheet, "Valuable vs Non-Valuable Summary", value_type_summary, row)

        # SVL (3 blocks)
        svl_sheet = safe_sheet_name(prefix, "SVL")
        row = 0
        row = write_block(writer, svl_sheet, "Source Summary", source_summary, row)
        row = write_block(writer, svl_sheet, "Source -> Vendor -> Location Drilldown", svl_drilldown, row)
        row = write_block(writer, svl_sheet, "Hierarchy Material / Category Mix", svl_mix, row)

        # Driver
        driver_analysis.to_excel(writer, sheet_name=safe_sheet_name(prefix, "Driver"), index=False)

        # Vehicle
        vehicle_analysis.to_excel(writer, sheet_name=safe_sheet_name(prefix, "Vehicle"), index=False)

# ============================================================
# 4) BASIC EXCEL FORMATTING
# ============================================================

wb = load_workbook(output_file)

header_fill = PatternFill(fill_type="solid", start_color="D9EAF7", end_color="D9EAF7")
header_font = Font(bold=True)
center_align = Alignment(horizontal="center", vertical="center")

for ws in wb.worksheets:
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    # Style the first row only
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = center_align

    # Auto-fit columns
    for col_cells in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col_cells[0].column)
        for cell in col_cells:
            try:
                val = str(cell.value) if cell.value is not None else ""
                if len(val) > max_len:
                    max_len = len(val)
            except:
                pass
        ws.column_dimensions[col_letter].width = min(max(max_len + 2, 12), 45)

wb.save(output_file)

print(f"✅ Revised Excel file created: {output_file}")
files.download(output_file)

📂 Upload one or more Inward cleaned CSV files...


Saving Inward-2_cleaned.csv to Inward-2_cleaned.csv

🔄 Processing file: Inward-2_cleaned.csv
✅ Revised Excel file created: Inward_Analytics_Output_Revised.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>